# Full run of `salmonella-wgs-amr` in Google Colab

Runs the whole pipeline on the **real public data** (SRR18072959): QC → fastp → SPAdes → QUAST (+ comparison with LT2)
→ read mapping and contig QC → Prokka → AMRFinderPlus.

**How to use it:** `Runtime → Run all`, and upload `salmonella-wgs-amr.zip` when asked.
It takes about **1.5–2 h** (SPAdes is the long step). Keep the tab open.
At the end, cell 9 prints a summary: copy it and send it to Claude. If a cell fails, send the last lines of the error.

In [ ]:
# 1. Upload and unzip the repository
import os, zipfile, glob
from google.colab import files

os.chdir('/content')
uploaded = files.upload()
zip_name = next(n for n in uploaded if n.endswith('.zip'))
zipfile.ZipFile(zip_name).extractall('/content/repo_zip')
hit = glob.glob('/content/repo_zip/**/scripts/05_contig_qc.py', recursive=True)
assert hit, 'scripts/05_contig_qc.py not found in the zip'
REPO = os.path.dirname(os.path.dirname(hit[0]))
os.chdir(REPO)
os.environ['REPO'] = REPO
os.environ['MAMBA_ROOT_PREFIX'] = '/content/micromamba'
os.environ['CONFIG'] = 'config/colab.sh'
# Colab: 2 CPUs and ~12 GB RAM
open(f'{REPO}/config/colab.sh', 'w').write('source config/config.sh\nTHREADS=2\nSPADES_MEM_GB=11\n')
print('Repository at:', REPO, '| CPUs:', os.cpu_count())

In [ ]:
%%bash
# 2. Install micromamba and both conda environments (~10 min)
set -euo pipefail
cd "$REPO"
command -v micromamba >/dev/null || curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /usr/local bin/micromamba
echo '== salmonella-assembly';   micromamba create -y -n salmonella-assembly   -f envs/assembly.yml   2>&1 | tail -5
echo '== salmonella-annotation'; micromamba create -y -n salmonella-annotation -f envs/annotation.yml 2>&1 | tail -5
micromamba run -n salmonella-assembly bash -c 'fastqc --version; fastp --version 2>&1; spades.py --version; quast.py --version; samtools --version | head -1'
micromamba run -n salmonella-annotation bash -c 'prokka --version 2>&1; amrfinder --version'

In [ ]:
%%bash
# 3. Download the reads (ENA) and the LT2 reference (NCBI) (~5 min)
set -euo pipefail
cd "$REPO"
micromamba run -n salmonella-assembly bash scripts/00_download_data.sh

In [ ]:
%%bash
# 4. Steps 1-5: QC, trimming, SPAdes assembly, QUAST, read mapping and contig QC (~1-1.5 h)
set -euo pipefail
cd "$REPO"
unset MPLBACKEND
micromamba run -n salmonella-assembly bash scripts/run_all.sh assembly 2>&1 | grep -vE '^(Approx|Started analysis|Analysis complete)'

In [ ]:
# 5. Contig QC plot (length vs depth, coloured by GC)
from IPython.display import Image, display
display(Image(f'{REPO}/results/05_contig_qc/contig_length_vs_depth.png', width=700))

In [ ]:
%%bash
# 6. Steps 6-7: Prokka annotation and AMRFinderPlus (~15-20 min)
set -euo pipefail
cd "$REPO"
micromamba run -n salmonella-annotation bash scripts/run_all.sh annotation

In [ ]:
%%bash
# 7. (Comparison) Prokka on ALL contigs, as in the original exercise
set -euo pipefail
cd "$REPO"
micromamba run -n salmonella-annotation env ANNOTATE_ALL=1 PROKKA_OUT=results/06_prokka_all_contigs \
    bash scripts/06_annotate.sh | tail -12

In [ ]:
%%bash
# 8. Quick taxonomic check of the contigs that the filter removed (largest 20, BLAST against NCBI nt)
cd "$REPO"
pip -q install biopython > /dev/null
python - <<'EOF' || echo 'BLAST step failed (NCBI busy?) - not essential, continue'
import pandas as pd, time
from Bio.Blast import NCBIWWW, NCBIXML
t = pd.read_csv('results/05_contig_qc/contig_table.tsv', sep='\t')
seqs = {}
name = None
for line in open('results/03_spades/contigs.fasta'):
    line = line.strip()
    if line.startswith('>'): name = line[1:].split()[0]; seqs[name] = []
    else: seqs[name].append(line)
removed = t[(~t.keep) & (t.length >= 1000)].sort_values('length', ascending=False).head(20)
query = ''.join(f'>{c}\n{"".join(seqs[c])[:3000]}\n' for c in removed.contig)
print(f'BLASTing {len(removed)} removed contigs (>= 1 kb, first 3 kb of each) against nt... (a few minutes)')
try:
    res = NCBIWWW.qblast('blastn', 'nt', query, megablast=True, hitlist_size=1, expect=1e-20)
    rows = []
    for rec in NCBIXML.parse(res):
        top = rec.alignments[0].hit_def[:70] if rec.alignments else 'no hit'
        rows.append((rec.query.split()[0], top))
    out = removed.merge(pd.DataFrame(rows, columns=['contig', 'top_hit']), on='contig')
    out.to_csv('results/05_contig_qc/removed_contigs_blast.tsv', sep='\t', index=False)
    print(out[['contig', 'length', 'gc', 'meandepth', 'top_hit']].to_string(index=False))
except Exception as e:
    print('BLAST failed:', e)
EOF

In [ ]:
%%bash
# 9. SUMMARY — copy everything below and send it to Claude
cd "$REPO"
echo '### fastp';      python -c "import json;j=json.load(open('results/02_trimmed/SRR18072959.fastp.json'));print(j['summary']['before_filtering']);print(j['summary']['after_filtering']);print(j['filtering_result']);print('dup',j['duplication'])"
echo '### SPAdes';     ls results/03_spades | grep '^K' | tr '\n' ' '; echo; grep -c '>' results/03_spades/contigs.fasta; grep -m1 -i 'SPAdes version' results/03_spades/spades.log
echo '### QUAST all';  cat results/04_assembly_qc/quast/report.txt | sed -n '1,40p'
echo '### QUAST vs LT2'; grep -E 'contigs|Largest|Total length|GC|N50|N90|L50|L90|Genome fraction|Unaligned|misassemblies|Duplication' results/04_assembly_qc/quast_vs_LT2/report.txt
echo '### mapping';    cat results/04_assembly_qc/flagstat.txt | head -8; cat results/04_assembly_qc/expected_coverage.txt
echo '### contig QC';  cat results/05_contig_qc/summary.tsv
echo '### Prokka filtered'; cat results/06_prokka/SRR18072959.txt
echo '### Prokka all contigs'; cat results/06_prokka_all_contigs/SRR18072959.txt 2>/dev/null
echo '### AMRFinder';  cat results/07_amrfinder/versions.txt; cut -f2-12 results/07_amrfinder/SRR18072959.amrfinder.tsv
echo '### BLAST removed'; cat results/05_contig_qc/removed_contigs_blast.tsv 2>/dev/null | cut -f1,2,3,4,8 | head -25

In [ ]:
# 10. Download the key results (figures, reports, tables)
!cd "$REPO" && zip -qr /content/salmonella_results.zip results/04_assembly_qc/quast results/04_assembly_qc/quast_vs_LT2/report.* results/05_contig_qc results/06_prokka/*.txt results/07_amrfinder results/02_trimmed/*.html -x '*.fasta'
files.download('/content/salmonella_results.zip')